# One Model, Three Jobs: the tabular foundation model idea, hands-on
**Companion to the Medium post on LimiX-2 (released 16 Sep 2026).**

LimiX-2's headline claim: a single 400M-parameter pretrained model does **classification**,
**regression**, and **missing-value imputation** in one forward pass, with no fine-tuning —
TabArena Elo **1935** (+117.4 above TabFM+), beating AutoGluon 1.6 on all three benchmark suites.

We can't run LimiX-2's 400M parameters here, so this notebook does the next best thing:
it builds the *messy spreadsheet* LimiX-2 is designed for, runs the **old way**
(three separate sklearn pipelines), then shows the **unified API shape** LimiX-2 offers —
one object, three jobs. The final section has the real LimiX-2 loading code for when you
have the checkpoint (non-commercial license).


In [ ]:
df = make_table()
print("rows:", len(df))
print("missing-value rate per feature:")
print(df[["age","income","tenure","city"]].isna().mean().round(3).to_string())
print(df.head(3).to_string())


rows: 3000
missing-value rate per feature:
age       0.148
income    0.153
tenure    0.154
city      0.147
    age         income  tenure     city  churn        spend
0  22.0  102295.783790    93.0  seattle      0  3110.492505
1  58.0            NaN    80.0   austin      0  2869.410180
2  52.0   73857.003166   111.0   austin      0  3386.351349


## Job 1 — classification: will the customer churn? (old way: its own tuned pipeline)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df[["age","income","tenure","city"]]; y = df["churn"]
num = ["age","income","tenure"]; cat = ["city"]
pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])
clf = Pipeline([("pre", pre), ("model", HistGradientBoostingClassifier(random_state=0))])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
clf.fit(Xtr, ytr)
print("churn classifier accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))


churn classifier accuracy: 0.965


## Job 2 — regression: what will the customer spend? (old way: a *second* pipeline)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

yr = df["spend"]
reg = Pipeline([("pre", pre), ("model", HistGradientBoostingRegressor(random_state=0))])
Xtr, Xte, ytr, yte = train_test_split(X, yr, test_size=0.2, random_state=0)
reg.fit(Xtr, ytr)
p = reg.predict(Xte)
print("spend RMSE:", round(mean_squared_error(yte, p) ** 0.5, 1), " R2:", round(r2_score(yte, p), 3))


spend RMSE: 349.5  R2: 0.846


## Job 3 — imputation: fill the blanks (old way: a *third* step, often an afterthought)

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

complete = df[["age","income","tenure"]].dropna().to_numpy()
rng2 = np.random.default_rng(1)
hide = rng2.random(complete.shape) < 0.10
corrupted = complete.copy(); corrupted[hide] = np.nan
mean_filled = np.where(np.isnan(corrupted), np.nanmean(corrupted, axis=0), corrupted)
iter_filled = IterativeImputer(random_state=0, max_iter=10).fit_transform(corrupted)
rmse = lambda a: float(np.sqrt(np.mean((a[hide] - complete[hide]) ** 2)))
print("mean imputation RMSE:     ", round(rmse(mean_filled), 1))
print("iterative imputation RMSE:", round(rmse(iter_filled), 1))


mean imputation RMSE:      11954.9
iterative imputation RMSE: 9083.8


## The LimiX-2 shape: one object, three jobs
Count the artifacts above: a classifier pipeline, a regressor pipeline, an imputer —
three things to train, version, and monitor. LimiX-2 collapses them into one frozen model.
Below is the same collapse with ordinary tools, to show the *interface* difference:
one object that answers all three questions about a messy table.


In [ ]:
class SingleTableModel:
    """One interface, three jobs: the API shape LimiX-2 offers natively."""
    def __init__(self, classifier, regressor, imputer):
        self.classifier, self.regressor, self.imputer = classifier, regressor, imputer
    def predict_label(self, table):   # classification
        return self.classifier.predict(table)
    def predict_value(self, table):   # regression
        return self.regressor.predict(table)
    def fill_missing(self, table):    # imputation
        t = table.copy()
        t[["age","income","tenure"]] = self.imputer.transform(t[["age","income","tenure"]])
        t["city"] = t["city"].fillna(t["city"].mode()[0])
        return t

model = SingleTableModel(clf, reg, IterativeImputer(random_state=0, max_iter=10).fit(X[["age","income","tenure"]]))
sample = X.head(5)
print("churn labels :", model.predict_label(sample).tolist())
print("spend values :", [round(v, 1) for v in model.predict_value(sample)])
print("missing left after fill_missing:", int(model.fill_missing(X).isna().sum().sum()))


churn labels : [0, 0, 0, 0, 0]
spend values : [3058.3, 2888.0, 3443.0, 2351.5, 2954.8]
missing left after fill_missing: 0


## The real thing: loading LimiX-2
The cell below is the actual LimiX-2 workflow from the Stable AI repo
(`github.com/limix-ldm-ai/LimiX`, checkpoint `LimiX-2.ckpt`, released 16 Sep 2026).
**Not executed here** — it needs the checkpoint, their inference code, and a GPU/CPU
with ~2GB free. Note the license: StableAI LimiX Non-Commercial License v1.0,
so this is research use unless you clear commercial terms.


In [ ]:
# --- requires: git clone https://github.com/limix-ldm-ai/LimiX.git ---
# from limix import LimiX2  # inference entry point from the repo
#
# model = LimiX2.from_pretrained("LimiX-2.ckpt")   # 400M params, no fine-tuning needed
# train_rows = df.dropna(subset=["churn"]).head(200)   # context = the prompt
# query_rows = df[df["churn"].isna()].head(50)          # rows to score
#
# churn_pred = model.classify(train_rows, query_rows, target="churn")
# spend_pred = model.regress(train_rows, query_rows, target="spend")
# filled    = model.impute(query_rows)                 # same forward-pass machinery
print("see markdown above: run this where the LimiX-2 checkpoint is available")


## Wrap-up
- Three sklearn pipelines did what one LimiX-2 forward pass does — the point of the post
  is the *collapse*: one frozen model, three jobs, zero retraining.
- The numbers behind the post (Stable AI, self-reported): TabArena Elo **1935**
  (+117.4 vs TabFM+ 1818), 94.5% win rate on 38 classification datasets,
  96.9% on 13 regression datasets; TALENT 1506, BCCO 1432.
- The transferable idea isn't 400M parameters — it's **training on synthetic data from
  structural causal models** instead of scraped tables.
- Open questions: independent replication, generalization to truly ugly production
  spreadsheets, and the non-commercial license.
